In [ ]:
import jax.numpy as jnp
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from vizopt.color import classical_mds, lab_to_rgb, optimize_colors

In [ ]:
words = ["paper", "Papier", "papel", "palpiri", "carta", "karatasi"]

In [ ]:
def edit_distance(a, b):
    """Levenshtein distance via DP."""
    a, b = a.lower(), b.lower()
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i, ca in enumerate(a, 1):
        prev = dp[:]
        dp[0] = i
        for j, cb in enumerate(b, 1):
            dp[j] = prev[j-1] if ca == cb else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]


In [ ]:
distances = pd.DataFrame(
    [[edit_distance(w1, w2) for w2 in words] for w1 in words],
    index=words,
    columns=words,
)
distances

In [ ]:
coords = classical_mds(distances)

# Uniform scaling: one scale factor for all dims so MDS distances are preserved.
# Per-dimension normalization would amplify near-zero eigenvalue dimensions,
# blowing up tiny coordinate differences into large color differences.
target_half_ranges = np.array([25, 50, 50])  # max half-extents for L, a, b
uniform_scale = (target_half_ranges / np.abs(coords).max(axis=0)).min()

Lab = coords * uniform_scale + np.array([55, 0, 0])  # center L at 55

colors = lab_to_rgb(Lab)

pd.DataFrame(Lab, index=words, columns=["L", "a", "b"]).round(1)

In [ ]:
def plot_colored_words(words, colors):
    fig, ax = plt.subplots(figsize=(7, 2))
    for i, (word, color) in enumerate(zip(words, colors)):
        ax.scatter(i, 0.15, color=color, s=500, zorder=3, edgecolors="0.3", linewidths=0.5)
        ax.text(i, -0.05, word, ha="center", va="top", fontsize=12)
    ax.set_xlim(-0.7, len(words) - 0.3)
    ax.set_ylim(-0.3, 0.45)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_colored_words(words, colors)


In [ ]:
main_color = "#784086"

In [ ]:
sparse_cb = lambda i, loss, *_: print(f"iter {i:4d}  loss={loss:.6f}") if i % 200 == 0 else None

colors, loss = optimize_colors(
    distances,
    fixed_colors={"paper": mcolors.to_rgb(main_color)},
    target_max_delta_e=50.0,
    learning_rate=0.05,
    n_iters=1000,
    callback=sparse_cb,
)
print(f"Final loss: {loss:.6f}")

In [ ]:
from scipy.stats import spearmanr
from vizopt.color import rgb_to_lab

n = len(words)
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
D = np.array(distances)

lab_opt = np.array(rgb_to_lab(jnp.array(colors)))
color_dists_opt = [np.linalg.norm(lab_opt[i] - lab_opt[j]) for i, j in pairs]
rho, _ = spearmanr(color_dists_opt, [D[i, j] for i, j in pairs])
print(f"Spearman ρ = {rho:.3f}")

plot_colored_words(words, colors)

In [ ]:
n_restarts = 10
results = []
for seed in range(n_restarts):
    c, loss = optimize_colors(
        distances,
        fixed_colors={"paper": mcolors.to_rgb(main_color)},
        target_max_delta_e=50.0,
        learning_rate=0.05,
        n_iters=1000,
        seed=seed,
    )
    results.append((loss, c))
    print(f"seed={seed:2d}  loss={loss:.6f}")

best_loss, best_colors = min(results, key=lambda x: x[0])
print(f"\nBest loss = {best_loss:.6f}")
plot_colored_words(words, best_colors)

In [ ]:
import networkx as nx

In [ ]:
tree = nx.balanced_tree(r=2, h=3, create_using=nx.DiGraph)
tree.nodes

In [ ]:
mapping = {}
queue = [(0, "")]
while queue:
    node, label = queue.pop(0)
    mapping[node] = label
    for i, child in enumerate(tree.successors(node)):
        queue.append((child, label + str(i)))

mapping[0] = "root"
tree = nx.relabel_nodes(tree, mapping)

In [ ]:
nx.draw_networkx(tree)